# 04 - CrewAI: Incident Response Crew

## Scenario: Northstar Incident Investigation

CrewAI is designed to model multi-agent interactions like a real-world company. You define **Agents** (who have roles and goals), **Tasks** (what needs to be done), and assemble them into a **Crew**.

In this notebook, we will build a Northstar Incident Response Crew. 
1. **The Analyst Agent**: Investigates logs to find the root cause of an outage.
2. **The Communications Agent**: Drafts a public status page update based on the Analyst's findings.

In [ ]:
import os
from crewai import Agent, Task, Crew, Process

# Make sure you have a dummy key set for local execution
os.environ.setdefault("OPENAI_API_KEY", "dummy-key")


## 1. Defining the Agents

Agents need a `role`, `goal`, and a `backstory` to help guide their system prompt.

In [ ]:
# Define the Log Analyst
analyst = Agent(
    role='Senior Site Reliability Engineer (SRE)',
    goal='Investigate system outages and identify root causes quickly.',
    backstory='You are a veteran SRE at Northstar. You excel at reading logs and finding the needle in the haystack.',
    verbose=True,
    allow_delegation=False
)

# Define the Communications Manager
comm_manager = Agent(
    role='Incident Communications Manager',
    goal='Translate technical incident details into clear, empathetic public status updates.',
    backstory='You manage Northstar\'s public status page. You must reassure customers without lying or hiding the truth.',
    verbose=True,
    allow_delegation=False
)


## 2. Defining the Tasks

Tasks are assigned to specific agents. The output of the first task automatically feeds into the second task.

In [ ]:
# Task 1: Investigate
investigate_task = Task(
    description='Review the following server logs for the EU-West region: [FATAL] Connection timeout to database at 10:04 UTC. Latency spiked to 8000ms. Root cause: Database index corruption.',
    expected_output='A short technical summary of the root cause.',
    agent=analyst
)

# Task 2: Draft Comms
draft_comms_task = Task(
    description='Using the technical summary, draft a 3-sentence update for the public status page. Apologize for the EU-West downtime.',
    expected_output='A 3-sentence public status update.',
    agent=comm_manager
)


## 3. Assembling the Crew

We assemble the Crew and set `process=Process.sequential`. This means Task 1 must finish before Task 2 starts.

In [ ]:
incident_crew = Crew(
    agents=[analyst, comm_manager],
    tasks=[investigate_task, draft_comms_task],
    process=Process.sequential
)


## 4. Running the Crew

*(Note: If you are running this without a valid OpenAI API key, the execution will fail. You can read the output structure below.)*

In [ ]:
try:
    print("🚨 [Incident Alert] Kicking off the Crew...")
    result = incident_crew.kickoff()
    print("\n\n✅ [Final Status Page Update]:")
    print(result)
except Exception as e:
    print(f"\nCrew Execution Failed (Check API Key): {e}")
    print("\n[Mock Output]")
    print("We are currently investigating elevated errors in the EU-West region caused by a database issue. Our engineering team is actively working to resolve this and restore service. We apologize for any disruption to your workflow.")


## Watch For

- **Sequential vs Hierarchical**: We used `Process.sequential` which is just a strict pipeline. CrewAI also supports `Process.hierarchical` where a "Manager Agent" dynamically delegates tasks to workers.
- **Over-delegation**: If you enable `allow_delegation=True`, agents can pass tasks back and forth. Be careful, as this can lead to infinite delegation loops if they get confused.

## Checkpoint

**1. In a sequential CrewAI process, how does the second Agent know what the first Agent discovered?**
- A) You have to manually write Python code to pass the variables.
- B) CrewAI automatically passes the `expected_output` of the first task as context to the second task.
- C) The agents communicate via a Slack integration.
- D) They don't; they are completely isolated.
